# Classificação de PII e máscaras de coluna (etapa 6.2)

Implementa o [ADR 0017](../../docs/adr/0017-classificacao-de-pii-e-mascaras-de-coluna.md) em três passos, todos guiados pelas **tags do catálogo**:

1. **Classificar:** completa `pii` e `pii_categoria` onde faltam (bronze, silver, quarentenas) e passa o cadastro do vendedor para `localizacao_comercial`.
2. **Criar as funções de máscara** `ops.mask_identificador`, `ops.mask_cidade` e `ops.mask_cep`, com o salt do hash guardado em `ops.governanca_config`.
3. **Aplicar** as máscaras nas colunas da gold que são dado pessoal do cliente.

| Coluna da gold | analista | gestor | dpo e engenharia |
|---|---|---|---|
| `customer_unique_id` | hash com salt | hash com salt | em claro |
| `customer_state` | em claro | em claro | em claro |
| `customer_city` | `NULL` | em claro | em claro |
| `customer_zip_code_prefix` | `NULL` | 3 primeiros dígitos | em claro |

**Parâmetro `modo`**

| Valor | O que faz |
|---|---|
| `simulacao` (padrão) | mostra o que faria, sem alterar nada |
| `executar` | classifica, cria funções, aplica máscaras e verifica |
| `reverter` | remove as máscaras (plano B se a atualização da gold reclamar delas) |
| `testar_perfil` | mostra o que o seu usuário enxerga agora e grava a verificação do perfil detectado |

> **Cuidado:** enquanto você **não** estiver em `olist_engenharia` (ou `olist_dpo`), qualquer atualização do pipeline da gold grava o dado já mascarado. Volte ao grupo antes de atualizar o pipeline.

In [0]:
# O QUE FAZ: define parâmetros, grupos e as regras de classificação e de máscara.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (configuração).
#
# FUNÇÕES NATIVAS E SQL:
# - dbutils.widgets.dropdown / dbutils.widgets.get: parâmetro `modo` no topo do notebook.
#
# REGRAS:
# - REGRAS_COLUNA: nome da coluna -> categoria de PII. Vendedor vira localizacao_comercial (ADR 0017, item 2).
# - COLUNAS_BRUTAS: colunas com a linha bruta do arquivo; só viram PII nas entidades que têm dado pessoal.
# - MASCARAS: coluna da gold -> função de máscara. A UF e o cadastro do vendedor ficam sem máscara de propósito.
from pyspark.sql import functions as F

CATALOG = "olist"
dbutils.widgets.dropdown("modo", "simulacao", ["simulacao", "executar", "reverter", "testar_perfil"], "Modo")
MODO = dbutils.widgets.get("modo")

GRUPOS = {"analista": "olist_analista", "gestor": "olist_gestor", "dpo": "olist_dpo", "engenharia": "olist_engenharia"}
SCHEMAS_CLASSIFICACAO = ["bronze", "silver", "quality", "gold"]

REGRAS_COLUNA = {
    "customer_unique_id": "identificador",
    "customer_city": "localizacao",
    "customer_state": "localizacao",
    "customer_zip_code_prefix": "localizacao",
    "seller_city": "localizacao_comercial",
    "seller_state": "localizacao_comercial",
    "seller_zip_code_prefix": "localizacao_comercial",
    "review_comment_message": "texto_livre",
    "review_comment_title": "texto_livre",
}
COLUNAS_BRUTAS = ("_corrupt_record", "_rescued_data")
ENTIDADES_COM_PII = ("customers", "sellers", "order_reviews")

MASCARAS = {
    "customer_unique_id": "mask_identificador",
    "customer_city": "mask_cidade",
    "customer_zip_code_prefix": "mask_cep",
}
FALHAS = []
print(f"modo = {MODO}")

In [0]:
# O QUE FAZ: cria (se não existir) a tabela quality.validacao_governanca e declara a função que grava resultados.
#
# FUNÇÕES DECLARADAS:
# - registrar(etapa, linhas, apagar_antes): apaga as linhas anteriores da etapa (ou dos perfis) e grava as novas em
#   quality.validacao_governanca. Rodar de novo troca o resultado, não duplica.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: cria a tabela de resultados uma vez, já com comentário em tabela e colunas.
# - DELETE FROM t WHERE ...: remove as linhas antigas (Delta). spark.createDataFrame(linhas, "schema DDL"): cria o
#   DataFrame a partir de tuplas. F.lit(x): coluna com valor fixo. F.current_timestamp(): hora da execução.
# - df.write.mode("append").saveAsTable(t): acrescenta as linhas à tabela.

VALIDACAO = f"{CATALOG}.quality.validacao_governanca"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDACAO} (
  etapa STRING COMMENT 'Etapa da Fase 6 que gerou a verificação (6.2-grants, 6.2-mascaras, 6.2-perfil, ...).',
  perfil STRING COMMENT 'Perfil avaliado (analista, gestor, dpo, engenharia) ou geral quando vale para todos.',
  verificacao STRING COMMENT 'O que foi verificado.',
  esperado STRING COMMENT 'Valor esperado pela regra.',
  obtido STRING COMMENT 'Valor encontrado no ambiente.',
  resultado STRING COMMENT 'OK, REPROVADA ou REGISTRO (só guarda um valor de referência).',
  executado_em TIMESTAMP COMMENT 'Momento da execução.'
)
COMMENT 'Resultado das verificações de governança e LGPD (grants, máscaras, filtros de linha, exclusão de cliente). Nenhuma linha guarda dado pessoal.'
""")


def registrar(etapa: str, linhas: list, apagar_antes: str = "etapa") -> None:
    """Grava as verificações. linhas = [(perfil, verificacao, esperado, obtido, resultado)]."""
    if apagar_antes == "etapa":
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}'")
    else:
        perfis = sorted({l[0] for l in linhas})
        lista = ", ".join(f"'{p}'" for p in perfis)
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}' AND perfil IN ({lista})")
    df = (spark.createDataFrame(linhas, "perfil string, verificacao string, esperado string, obtido string, resultado string")
          .withColumn("etapa", F.lit(etapa))
          .withColumn("executado_em", F.current_timestamp())
          .select("etapa", "perfil", "verificacao", "esperado", "obtido", "resultado", "executado_em"))
    df.write.mode("append").saveAsTable(VALIDACAO)

In [0]:
# O QUE FAZ: lê do catálogo as colunas, o tipo de cada objeto e as tags atuais, e calcula o que falta classificar.
#   Nada é alterado aqui.
#
# FUNÇÕES DECLARADAS:
# - tipo_de(s, t): tipo do objeto (MATERIALIZED_VIEW, VIEW, METRIC_VIEW, MANAGED, STREAMING_TABLE...).
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.columns / tables / column_tags / table_tags: colunas, objetos e tags do catálogo.
# - startswith(nome, 'x'): descarta as tabelas de apoio (__materialization_*) e o log do pipeline (event_log_*).
# - dict.setdefault(chave, {}): cria o dicionário interno na primeira vez. t.startswith(tupla): testa vários prefixos.
lista = ", ".join(f"'{s}'" for s in SCHEMAS_CLASSIFICACAO)
colunas = spark.sql(f"""
    SELECT table_schema, table_name, column_name FROM {CATALOG}.information_schema.columns
    WHERE table_schema IN ({lista}) AND NOT startswith(table_name, '__') AND NOT startswith(table_name, 'event_log_')
""").collect()
tipos = {(r["table_schema"], r["table_name"]): r["table_type"] for r in spark.sql(
    f"SELECT table_schema, table_name, table_type FROM {CATALOG}.information_schema.tables WHERE table_schema IN ({lista})").collect()}

tags_col = {}
for r in spark.sql(f"SELECT schema_name, table_name, column_name, tag_name, tag_value FROM {CATALOG}.information_schema.column_tags WHERE tag_name IN ('pii','pii_categoria')").collect():
    tags_col.setdefault((r["schema_name"], r["table_name"], r["column_name"]), {})[r["tag_name"]] = r["tag_value"]
tags_tab = {(r["schema_name"], r["table_name"]) for r in spark.sql(
    f"SELECT schema_name, table_name FROM {CATALOG}.information_schema.table_tags WHERE tag_name = 'contem_pii' AND tag_value = 'true'").collect()}


def tipo_de(s, t):
    return tipos.get((s, t), "TABLE")


desejado = {}
for s, t, c in ((r["table_schema"], r["table_name"], r["column_name"]) for r in colunas):
    if tipo_de(s, t) in ("VIEW", "METRIC_VIEW"):
        continue
    if c in REGRAS_COLUNA:
        cat = REGRAS_COLUNA[c]
    elif c in COLUNAS_BRUTAS and t.startswith(ENTIDADES_COM_PII):
        cat = "registro_bruto"
    else:
        continue
    if s == "gold" and (s, t, c) not in tags_col:
        continue  # na gold só se recategoriza o que a Fase 4 já tinha marcado
    desejado[(s, t, c)] = cat

mudar_coluna = {k: cat for k, cat in desejado.items() if tags_col.get(k, {}).get("pii") != "true" or tags_col[k].get("pii_categoria") != cat}
mudar_tabela = sorted({(s, t) for (s, t, c) in desejado} - tags_tab)
print(f"colunas de PII previstas: {len(desejado)}; a (re)classificar: {len(mudar_coluna)}; tabelas sem contem_pii=true: {len(mudar_tabela)}")
for (s, t, c), cat in sorted(mudar_coluna.items())[:60]:
    print(f"  {s}.{t}.{c}: {tags_col.get((s, t, c), {}).get('pii_categoria', 'sem tag')} -> {cat}")

In [0]:
# O QUE FAZ: aplica as tags que faltam (colunas e tabelas). Só no modo `executar`. Falhas são juntadas em FALHAS e
#   reportadas no fim, sem parar o laço.
#
# FUNÇÕES DECLARADAS:
# - verbo(s, t): palavra do ALTER conforme o tipo do objeto (VIEW, MATERIALIZED VIEW ou TABLE).
# - tentar(descricao, principal, alternativa): executa o comando; se falhar, tenta a alternativa; registra a falha.
#
# FUNÇÕES NATIVAS E SQL:
# - ALTER TABLE t ALTER COLUMN c SET TAGS ('chave' = 'valor'): grava as tags da coluna (Unity Catalog).
# - ALTER TABLE t SET TAGS (...): grava as tags da tabela. spark.sql(texto): executa o SQL.
def verbo(s, t):
    tp = tipo_de(s, t)
    return "VIEW" if tp == "VIEW" else ("MATERIALIZED VIEW" if tp == "MATERIALIZED_VIEW" else "TABLE")


def tentar(descricao, principal, alternativa=None):
    for sql in [x for x in (principal, alternativa) if x]:
        try:
            spark.sql(sql)
            print("ok   ", descricao)
            return True
        except Exception as e:
            ultimo = str(e)[:250]
    FALHAS.append((descricao, ultimo))
    print("FALHA", descricao, "->", ultimo[:160])
    return False


if MODO == "executar":
    for (s, t, c), cat in sorted(mudar_coluna.items()):
        alvo = f"{CATALOG}.{s}.{t}"
        tags = f"('pii' = 'true', 'pii_categoria' = '{cat}')"
        tentar(f"tag {s}.{t}.{c}", f"ALTER {verbo(s, t)} {alvo} ALTER COLUMN {c} SET TAGS {tags}",
               f"ALTER TABLE {alvo} ALTER COLUMN {c} SET TAGS {tags}")
    for s, t in mudar_tabela:
        alvo = f"{CATALOG}.{s}.{t}"
        tentar(f"contem_pii {s}.{t}", f"ALTER {verbo(s, t)} {alvo} SET TAGS ('contem_pii' = 'true')",
               f"ALTER TABLE {alvo} SET TAGS ('contem_pii' = 'true')")
else:
    print("Nada alterado (modo", MODO + ").")

In [0]:
# O QUE FAZ: cria a tabela de configuração do salt, gera o salt na primeira vez e (re)cria as 3 funções de máscara.
#   O salt NUNCA é impresso. Só no modo `executar`.
#
# FUNÇÕES DECLARADAS:
# - corpo_*: texto SQL do corpo de cada função, montado com os nomes dos grupos (e o salt, no identificador).
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ops.governanca_config: guarda o salt (sem acesso para consumidores).
# - uuid(): identificador aleatório; dois uuid() concatenados formam o salt. INSERT INTO ... SELECT: grava uma vez.
# - CREATE OR REPLACE FUNCTION ... RETURN CASE WHEN ...: função SQL usada como máscara.
# - is_account_group_member('grupo'): verdadeiro se o usuário que consulta está no grupo de conta.
# - sha2(texto, 256): hash SHA-256 (64 caracteres). concat(a, b): junta o salt ao valor antes do hash.
# - substr(v, 1, 3): três primeiros caracteres (o CEP truncado do gestor).
DPO, ENG, GES = GRUPOS["dpo"], GRUPOS["engenharia"], GRUPOS["gestor"]
CONFIG = f"{CATALOG}.ops.governanca_config"

if MODO == "executar":
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {CONFIG} (
        chave STRING COMMENT 'Nome do parâmetro.', valor STRING COMMENT 'Valor do parâmetro (o salt do hash de PII).',
        atualizado_em TIMESTAMP COMMENT 'Quando o parâmetro foi gravado.')
        COMMENT 'Parâmetros de governança. Sem acesso para consumidores: guarda o salt do hash de PII (ADR 0017).'""")
    spark.sql(f"""INSERT INTO {CONFIG}
        SELECT 'salt_pii', concat(uuid(), uuid()), current_timestamp()
        WHERE NOT EXISTS (SELECT 1 FROM {CONFIG} WHERE chave = 'salt_pii')""")
    salt = spark.sql(f"SELECT valor FROM {CONFIG} WHERE chave = 'salt_pii'").first()[0]

    privilegiado = f"is_account_group_member('{DPO}') OR is_account_group_member('{ENG}')"
    corpo_id = f"CASE WHEN {privilegiado} THEN v ELSE sha2(concat('{salt}', v), 256) END"
    corpo_cidade = f"CASE WHEN {privilegiado} OR is_account_group_member('{GES}') THEN v ELSE NULL END"
    corpo_cep = (f"CASE WHEN {privilegiado} THEN v WHEN is_account_group_member('{GES}') THEN substr(v, 1, 3) ELSE NULL END")

    funcoes = {
        "mask_identificador": ("hash com salt para analista e gestor; em claro para dpo e engenharia", corpo_id),
        "mask_cidade": ("NULL para analista; em claro para gestor, dpo e engenharia", corpo_cidade),
        "mask_cep": ("NULL para analista; 3 primeiros dígitos para gestor; em claro para dpo e engenharia", corpo_cep),
    }
    for nome, (comentario, corpo) in funcoes.items():
        spark.sql(f"CREATE OR REPLACE FUNCTION {CATALOG}.ops.{nome}(v STRING) RETURNS STRING COMMENT '{comentario}' RETURN {corpo}")
        print("função criada:", nome)
    spark.sql(f"COMMENT ON TABLE {CONFIG} IS 'Parâmetros de governança. Sem acesso para consumidores: guarda o salt do hash de PII (ADR 0017).'")
else:
    print("Nada alterado (modo", MODO + ").")

In [0]:
# O QUE FAZ: aplica (executar) ou remove (reverter) as máscaras nas colunas da gold que são dado pessoal do cliente.
#   No modo `simulacao` só lista o que faria. Compara com o que já existe para não repetir.
#
# FUNÇÕES DECLARADAS:
# - mascaras_atuais(): lê information_schema.column_masks e devolve {(schema, tabela, coluna): nome da função}.
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.column_masks: máscaras ativas. As colunas são lidas por posição (catálogo, schema, tabela,
#   coluna, função) porque o nome das últimas colunas varia.
# - ALTER MATERIALIZED VIEW / ALTER TABLE ... ALTER COLUMN c SET MASK função: liga a máscara à coluna.
# - ... ALTER COLUMN c DROP MASK: remove a máscara. split("."): pega o último pedaço do nome completo da função.
def mascaras_atuais():
    df = spark.sql(f"SELECT * FROM {CATALOG}.information_schema.column_masks")
    cols = df.columns
    if len(cols) < 5:
        raise RuntimeError(f"column_masks com colunas inesperadas: {cols}")
    return {(r[cols[1]], r[cols[2]], r[cols[3]]): str(r[cols[4]]).split(".")[-1] for r in df.collect()}


alvos = sorted((s, t, c) for (s, t, c) in desejado if s == "gold" and c in MASCARAS)
atuais = mascaras_atuais()
print("colunas-alvo:", [f"{t}.{c}" for s, t, c in alvos])

if MODO in ("simulacao", "executar"):
    for s, t, c in alvos:
        funcao = MASCARAS[c]
        if atuais.get((s, t, c)) == funcao:
            print("já tem   ", f"{s}.{t}.{c} <- {funcao}")
            continue
        if MODO == "simulacao":
            print("[simulacao]", f"{s}.{t}.{c} <- {funcao}")
            continue
        alvo = f"{CATALOG}.{s}.{t}"
        tentar(f"máscara {s}.{t}.{c}",
               f"ALTER {verbo(s, t)} {alvo} ALTER COLUMN {c} SET MASK {CATALOG}.ops.{funcao}",
               f"ALTER TABLE {alvo} ALTER COLUMN {c} SET MASK {CATALOG}.ops.{funcao}")
elif MODO == "reverter":
    for (s, t, c), funcao in sorted(atuais.items()):
        if funcao in MASCARAS.values():
            alvo = f"{CATALOG}.{s}.{t}"
            tentar(f"remover máscara {s}.{t}.{c}", f"ALTER {verbo(s, t)} {alvo} ALTER COLUMN {c} DROP MASK",
                   f"ALTER TABLE {alvo} ALTER COLUMN {c} DROP MASK")

if FALHAS:
    raise RuntimeError(f"{len(FALHAS)} comando(s) falharam: {FALHAS}")

In [0]:
# O QUE FAZ: verifica o resultado e grava em quality.validacao_governanca (etapa 6.2-mascaras). Só no modo `executar`.
#   Verificações: (1) cada coluna-alvo tem a máscara esperada; (2) nenhuma coluna da gold com nome de dado pessoal
#   do cliente ficou sem máscara; (3) toda coluna que as regras classificam como PII nas camadas bronze, silver e
#   quality tem as tags; (4) o salt existe e as 3 funções existem.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (reaproveita mascaras_atuais, tipo_de e registrar).
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.routines: funções do catálogo. information_schema.column_tags: tags atuais (relidas, porque
#   a classificação acabou de mudar). sorted(), set(): comparação entre o esperado e o encontrado.
if MODO != "executar":
    print("Verificação só roda no modo executar.")
else:
    atuais = mascaras_atuais()
    linhas = []

    ok = [(s, t, c) for (s, t, c) in alvos if atuais.get((s, t, c)) == MASCARAS[c]]
    linhas.append(("geral", "colunas-alvo da gold com a máscara esperada", str(len(alvos)), str(len(ok)),
                   "OK" if len(ok) == len(alvos) and alvos else "REPROVADA"))

    na_gold = [(r["table_schema"], r["table_name"], r["column_name"]) for r in colunas
               if r["table_schema"] == "gold" and r["column_name"] in MASCARAS and tipo_de(r["table_schema"], r["table_name"]) not in ("VIEW", "METRIC_VIEW")]
    sem_mascara = [f"{t}.{c}" for (s, t, c) in na_gold if atuais.get((s, t, c)) != MASCARAS[c]]
    linhas.append(("geral", "nenhuma coluna de dado pessoal do cliente na gold sem máscara", "0", str(len(sem_mascara)) + (f": {sem_mascara[:3]}" if sem_mascara else ""),
                   "OK" if not sem_mascara else "REPROVADA"))

    tags_novas = {}
    for r in spark.sql(f"SELECT schema_name, table_name, column_name, tag_name, tag_value FROM {CATALOG}.information_schema.column_tags WHERE tag_name IN ('pii','pii_categoria')").collect():
        tags_novas.setdefault((r["schema_name"], r["table_name"], r["column_name"]), {})[r["tag_name"]] = r["tag_value"]
    sem_tag = [f"{s}.{t}.{c}" for (s, t, c), cat in desejado.items()
               if tags_novas.get((s, t, c), {}).get("pii") != "true" or tags_novas[(s, t, c)].get("pii_categoria") != cat]
    linhas.append(("geral", "colunas de PII classificadas com a categoria prevista (bronze, silver, quality e gold)", "0 sem tag", f"{len(sem_tag)} sem tag" + (f": {sem_tag[:3]}" if sem_tag else ""),
                   "OK" if not sem_tag else "REPROVADA"))

    fn = {r["routine_name"] for r in spark.sql(f"SELECT routine_name FROM {CATALOG}.information_schema.routines WHERE routine_schema = 'ops'").collect()}
    linhas.append(("geral", "funções de máscara existem em ops", ", ".join(sorted(MASCARAS.values())), ", ".join(sorted(fn & set(MASCARAS.values()))),
                   "OK" if set(MASCARAS.values()) <= fn else "REPROVADA"))

    salt_ok = spark.sql(f"SELECT count(*) FROM {CONFIG} WHERE chave = 'salt_pii' AND length(valor) >= 32").first()[0]
    linhas.append(("geral", "salt do hash gravado em ops.governanca_config (valor não exibido)", "1", str(salt_ok), "OK" if salt_ok == 1 else "REPROVADA"))

    registrar("6.2-mascaras", linhas)
    for l in linhas:
        print(f"[{l[4]:9}] {l[1]} | obtido: {l[3][:140]}")
    if FALHAS or any(l[4] == "REPROVADA" for l in linhas):
        raise RuntimeError(f"verificação reprovada ou comandos com falha: {FALHAS}; veja {VALIDACAO}")
    print("\nclassificação e máscaras conferidas")

In [0]:
# O QUE FAZ: modo `testar_perfil`. Descobre a quais grupos do projeto o seu usuário pertence agora, lê a gold como
#   esse perfil e confere se o que aparece bate com a matriz de máscaras. Grava o resultado por perfil (etapa 6.2-perfil).
#   Rode uma vez para cada perfil: troque a sua participação nos grupos na interface, reinicie a sessão e rode de novo.
#
# FUNÇÕES DECLARADAS:
# - pertence(grupo): True se o usuário que roda o notebook está no grupo de conta.
# - verificar(...): monta uma linha de resultado comparando esperado e obtido.
#
# FUNÇÕES NATIVAS E SQL:
# - is_account_group_member('grupo'): mesma função usada nas máscaras.
# - count(coluna): conta só os valores não nulos. count(DISTINCT coluna): clientes distintos. max(length(coluna)):
#   tamanho do maior texto (32 = identificador em claro, 64 = hash; 5 = CEP inteiro, 3 = truncado).
# - .first(): primeira linha do resultado. ORDER BY ... DESC LIMIT 1: o valor de referência mais recente.
if MODO != "testar_perfil":
    print("Rode com modo = testar_perfil.")
else:
    def pertence(grupo: str) -> bool:
        return bool(spark.sql(f"SELECT is_account_group_member('{grupo}')").first()[0])

    membros = {papel: pertence(g) for papel, g in GRUPOS.items()}
    perfil = next((p for p in ("engenharia", "dpo", "gestor", "analista") if membros[p]), "nenhum")
    efetivo = "analista" if perfil == "nenhum" else perfil
    print("participação:", membros, "-> perfil efetivo:", perfil)

    dim = spark.sql(f"""
        SELECT count(*) AS linhas, max(length(customer_unique_id)) AS len_id, count(customer_city) AS cidades,
               count(customer_zip_code_prefix) AS ceps, max(length(customer_zip_code_prefix)) AS len_cep,
               count(customer_state) AS ufs
        FROM {CATALOG}.gold.dim_customer""").first()
    fato = spark.sql(f"SELECT count(*) AS pedidos, count(DISTINCT customer_unique_id) AS clientes FROM {CATALOG}.gold.fact_orders").first()

    claro = efetivo in ("engenharia", "dpo")
    linhas = []

    def verificar(nome, esperado, obtido, ok):
        linhas.append((perfil, nome, str(esperado), str(obtido), "OK" if ok else "REPROVADA"))

    verificar("customer_unique_id: tamanho do maior valor (32 em claro, 64 hash)", 32 if claro else 64, dim["len_id"], dim["len_id"] == (32 if claro else 64))
    verificar("customer_city: valores visíveis", ">0" if efetivo != "analista" else 0, dim["cidades"], (dim["cidades"] > 0) if efetivo != "analista" else dim["cidades"] == 0)
    esp_cep = {"engenharia": 5, "dpo": 5, "gestor": 3}.get(efetivo, None)
    verificar("customer_zip_code_prefix: tamanho do maior valor (5 inteiro, 3 truncado, vazio para analista)", esp_cep if esp_cep else "nenhum",
              dim["len_cep"] if dim["ceps"] else "nenhum", (dim["len_cep"] == esp_cep) if esp_cep else dim["ceps"] == 0)
    verificar("customer_state: sempre em claro", ">0", dim["ufs"], dim["ufs"] > 0)

    if claro:
        linhas.append((perfil, "referencia_clientes_distintos (valor de comparação para os outros perfis)", "", str(fato["clientes"]), "REGISTRO"))
        linhas.append((perfil, "referencia_pedidos", "", str(fato["pedidos"]), "REGISTRO"))
    else:
        ref = {r["verificacao"]: r["obtido"] for r in spark.sql(f"""
            SELECT verificacao, obtido FROM {VALIDACAO}
            WHERE etapa = '6.2-perfil' AND resultado = 'REGISTRO' AND perfil IN ('engenharia', 'dpo')
            ORDER BY executado_em DESC""").collect()[::-1]}
        r_cli, r_ped = ref.get("referencia_clientes_distintos (valor de comparação para os outros perfis)"), ref.get("referencia_pedidos")
        if r_cli is None:
            linhas.append((perfil, "clientes distintos iguais aos do perfil em claro", "rodar antes como dpo ou engenharia", str(fato["clientes"]), "REGISTRO"))
        else:
            verificar("clientes distintos iguais aos do perfil em claro (o hash preserva a contagem)", r_cli, fato["clientes"], str(fato["clientes"]) == r_cli)
            verificar("pedidos iguais aos do perfil em claro (KPI não muda)", r_ped, fato["pedidos"], str(fato["pedidos"]) == r_ped)

    registrar("6.2-perfil", linhas, apagar_antes="perfil")
    for l in linhas:
        print(f"[{l[4]:9}] {l[1]} | esperado: {l[2]} | obtido: {l[3]}")
    amostra = spark.sql(f"SELECT customer_unique_id, customer_city, customer_state, customer_zip_code_prefix FROM {CATALOG}.gold.dim_customer LIMIT 5")
    display(amostra)
    if any(l[4] == "REPROVADA" for l in linhas):
        raise RuntimeError(f"perfil {perfil}: verificação reprovada; veja {VALIDACAO}")

## O que olhar no resultado

- `simulacao`: a lista de colunas a classificar (bronze, silver, quarentenas e as de vendedor) e as colunas-alvo da máscara (`dim_customer.customer_unique_id`, `customer_city`, `customer_zip_code_prefix` e `fact_orders.customer_unique_id`).
- `executar`: `ok` em cada tag e máscara, e 5 verificações `[OK]`. Se a máscara em materialized view falhar nas duas sintaxes, a célula mostra a mensagem do erro: **guarde o texto** e me envie, porque aciona o plano B do ADR 0017 (views seguras).
- `testar_perfil`: 4 a 6 verificações por perfil e 5 linhas de amostra. Rode como `engenharia` (ou `dpo`) **primeiro**, para gravar o valor de referência de clientes distintos.
- Depois de aplicar as máscaras, **atualize o pipeline da gold** (você, em `olist_engenharia`) e rode `executar` de novo: ele confirma que as máscaras sobreviveram à atualização.